# Demo: IoMT Attack Detector

This is the demo of our best model, the **Sparse Autoencoder** (picked in `compare_models.ipynb`). It's a small web app made with Gradio: you pick a sample of network traffic from the test set, and the app shows whether the model thinks it's an attack, how unusual the sample is, what the sample really was, and which features looked the most abnormal.

I wanted it to make sense even for someone who doesn't know anything about our project, so:

- everything is explained in plain words, and the technical numbers (reconstruction error and threshold) are only shown small at the bottom
- the features have readable names (for example *Connection-request (SYN) signals* instead of `syn_flag_number`), and every attack type has a one-line explanation
- there's an **EN / عربي** switch and a **dark / light** mode button

The app doesn't load the model by itself: it uses `predict()` from `inference/inference.py`, so the demo goes through the same inference pipeline we deploy.

**How to run it:** Runtime → Run all. The last cell prints a public link (`share=True`) that we can open during the discussion. The link only works while this Colab session is running.

## 1. Setup

Colab doesn't come with Gradio, so I install it first, then get the latest version of our repo.

In [ ]:
!pip install -q gradio

import os
if not os.path.exists("iomt-anomaly-detection"):
    !git clone -q https://github.com/JanaAlzobidi/iomt-anomaly-detection.git
else:
    !git -C iomt-anomaly-detection pull -q     # get the latest version of the repo

## 2. Loading the model and the test data

I load the saved Sparse AE with `load_model()` from `inference.py` and score the whole test set once at the start. That way the app answers instantly, and I can also show the error of each feature (the bars in the result). I check that these scores match the ones `predict()` gives (up to tiny rounding differences, because Keras does the math in a slightly different order for one row than for thousands of rows). The result panel itself always uses `predict()`.

I also pick the 3 examples for the presentation automatically, so we don't have to search for good rows by hand:

- **normal:** a typical normal row that the model correctly calls normal
- **easy attack:** a typical *DDoS SYN Flood* row, which the model catches
- **hard attack:** a typical *ARP Spoofing* row that the model misses, to show the limits of our model honestly

"Typical" means the row with the median score of its group, so the example is not an extreme case. The numbers on the stat cards (F1, attacks caught, false alarms) are also computed here from the real predictions, not typed by hand.

In [ ]:
import sys
import numpy as np
from sklearn.metrics import f1_score

REPO = "iomt-anomaly-detection"
sys.path.insert(0, f"{REPO}/inference")
from inference import load_model, predict          # our inference pipeline (inference/inference.py)

test = np.load(f"{REPO}/data/test.npz", allow_pickle=True)
X_test = test["X"]
y_test = test["y"].astype(int)
attack_test = test["attack_type"].astype(str)
N_ROWS = len(X_test)

model, meta = load_model()
THRESHOLD = meta["threshold"]
FEATURES = meta["features"]

# score the whole test set once, so the demo answers instantly and can show the error of every feature
recon = model.predict(X_test, batch_size=4096, verbose=0)
ERRORS = np.square(X_test - recon)     # error per feature
SCORES = ERRORS.mean(axis=1)           # anomaly score per row
PRED = SCORES > THRESHOLD              # True = attack

# check that these scores match the ones predict() gives for a single row.
# Tiny differences are normal: Keras does the math in a slightly different order for 1 row
# than for 4,096 rows, and our scores are very small numbers, so they show up in the last digits.
check_rows = [0, 1234, 40000]
single = np.array([predict(X_test[i])[0]["score"] for i in check_rows])
rel_diff = np.max(np.abs(single - SCORES[check_rows]) / SCORES[check_rows])
print(f"Largest difference between predict() and the batch scores: {rel_diff:.2e} (relative)")
assert rel_diff < 1e-2, "predict() and the batch scores do not match"

benign = y_test == 0
STATS = {
    "f1": f1_score(y_test, PRED),
    "recall": PRED[~benign].mean(),          # share of attacks that were caught
    "false_alarms": PRED[benign].mean(),     # share of normal rows flagged as attacks
    "attack_types": len(set(attack_test[~benign])),
}


def typical_row(mask):
    # the row with the median score of a group, so the example is typical and not an extreme case
    rows = np.flatnonzero(mask)
    return int(rows[np.argmin(np.abs(SCORES[rows] - np.median(SCORES[rows])))])


EXAMPLES = {
    "normal": typical_row(benign & ~PRED),                                  # normal row, correctly normal
    "easy": typical_row((attack_test == "DDoS SYN Flood") & PRED),          # easy attack, caught
    "hard": typical_row((attack_test == "ARP Spoofing") & ~PRED),           # hard attack, missed
}
BENIGN_ROWS = np.flatnonzero(benign)
ATTACK_ROWS = np.flatnonzero(~benign)
RNG = np.random.default_rng()

print(f"Model loaded | threshold {THRESHOLD:.3g}")
print(f"F1 {STATS["f1"]:.3f} | attacks caught {STATS["recall"]:.1%} | false alarms {STATS["false_alarms"]:.2%}")
for key, row in EXAMPLES.items():
    print(f"{key:<7} row {row:>5} | {attack_test[row]:<15} | score {SCORES[row]:.2e} | predicted {"attack" if PRED[row] else "normal"}")

## 3. The texts in both languages

All the text of the app is in one dictionary, in English and Arabic, so switching the language just picks the other side. This cell also has a plain-language name for each of our 40 features and a short explanation for each of the 19 traffic types (normal + 18 attacks). The two `assert` lines at the end make sure nothing is missing.

In [ ]:
# all text of the demo, in English and Arabic

TEXT = {
    "en": {
        "title": "IoMT Attack Detector", "tag": "Deep learning final project · Sparse Autoencoder",
        "badge": "Live demo · trained on CICIoMT2024",
        "hero_title": "Is this medical device under attack?",
        "hero_sub": "Hospitals connect devices like heart monitors and insulin pumps to the network. "
                    "Our model reads their network traffic and warns when something looks like a cyberattack.",
        "s1h": "It learned what normal looks like", "s1": "Trained only on normal, everyday traffic.",
        "s2h": "It tries to rebuild each sample", "s2": "Squeezes the traffic into a short summary, then rebuilds it.",
        "s3h": "A big mistake means suspicious", "s3": "Can't rebuild it well? It has never seen anything like it.",
        "stats": ["F1 score", "of attacks caught", "false alarms", "attack types tested"],
        "pick": "Choose a traffic sample", "pick_sub": "An example, or any of our {n:,} test samples",
        "try": "TRY AN EXAMPLE",
        "ex": {"normal": ["Normal traffic", "Everyday device traffic"],
               "easy": ["Easy attack", "DDoS SYN Flood"],
               "hard": ["Hard attack", "ARP Spoofing"]},
        "num": "Sample number (0 to {last:,})", "check": "Check sample", "random": "Random sample",
        "to_light": "Light mode", "to_dark": "Dark mode",
        "sample": "Sample", "how": "How unusual is this sample?", "here": "This sample",
        "zone_ok": "Normal zone", "line": "Alarm line", "zone_bad": "Attack zone",
        "real": "WHAT IT REALLY WAS", "feat": "WHAT LOOKED MOST UNUSUAL",
        "att": "Attack detected", "att_sub": "This traffic doesn't look like normal device traffic.",
        "nor": "Looks normal", "nor_sub": "Nothing suspicious in this traffic.",
        "ok": "Model was right", "miss": "Model missed it", "false_alarm": "False alarm",
        "far_above": "{r:.0f}× above the alarm line", "above": "Just above the alarm line",
        "below": "Just below the alarm line", "far_below": "Well below the alarm line",
        "bad_row": "Type a whole number between 0 and {last:,}.",
        "footer": "CICIoMT2024 dataset · 40 network features · trained on normal traffic only",
    },
    "ar": {
        "title": "كاشف الهجمات على الأجهزة الطبية", "tag": "مشروع التعلم العميق النهائي · Sparse Autoencoder",
        "badge": "عرض مباشر · مدرّب على CICIoMT2024",
        "hero_title": "هل هذا الجهاز الطبي يتعرض لهجوم؟",
        "hero_sub": "المستشفيات توصّل أجهزة مثل أجهزة مراقبة القلب ومضخات الإنسولين بالشبكة. "
                    "الموديل حقنا يقرأ حركة الشبكة لهذه الأجهزة وينبّه إذا شاف شي يشبه هجوم إلكتروني.",
        "s1h": "تعلّم شكل الطبيعي", "s1": "تدرّب على الترافيك الطبيعي اليومي فقط.",
        "s2h": "يحاول يعيد بناء كل عينة", "s2": "يضغط الترافيك في ملخص قصير، ثم يعيد بناءه.",
        "s3h": "خطأ كبير = مشبوه", "s3": "ما قدر يعيد بناءها كويس؟ إذًا غير طبيعية.",
        "stats": ["مقياس F1", "من الهجمات انكشفت", "إنذارات كاذبة", "نوع هجوم تم اختباره"],
        "pick": "اختر عينة ترافيك", "pick_sub": "مثال جاهز، أو أي عينة من {n:,} عينة اختبار",
        "try": "جرّب مثال",
        "ex": {"normal": ["ترافيك طبيعي", "ترافيك يومي عادي"],
               "easy": ["هجوم سهل", "DDoS SYN Flood"],
               "hard": ["هجوم صعب", "ARP Spoofing"]},
        "num": "رقم العينة (من 0 إلى {last:,})", "check": "افحص العينة", "random": "عينة عشوائية",
        "to_light": "الوضع الفاتح", "to_dark": "الوضع الداكن",
        "sample": "عينة", "how": "قد إيش هالعينة غريبة؟", "here": "هذه العينة",
        "zone_ok": "منطقة الطبيعي", "line": "خط الإنذار", "zone_bad": "منطقة الهجوم",
        "real": "النوع الحقيقي", "feat": "وش أكثر شي كان غريب",
        "att": "تم كشف هجوم", "att_sub": "هذا الترافيك ما يشبه ترافيك الأجهزة الطبيعي.",
        "nor": "يبدو طبيعي", "nor_sub": "ما فيه شي مشبوه في هذا الترافيك.",
        "ok": "الموديل صح", "miss": "الموديل فاته", "false_alarm": "إنذار كاذب",
        "far_above": "أعلى من خط الإنذار بـ {r:.0f} مرة", "above": "فوق خط الإنذار بشوي",
        "below": "تحت خط الإنذار بشوي", "far_below": "تحت خط الإنذار بكثير",
        "bad_row": "اكتب رقم صحيح من 0 إلى {last:,}.",
        "footer": "بيانات CICIoMT2024 · 40 خاصية من الشبكة · مدرّب على الترافيك الطبيعي فقط",
    },
}

# plain-language names for the 40 features (same names as data/config.json)
FEATURE_NAMES = {
    "header_length": ["Packet header length", "طول ترويسة الحزمة"],
    "protocol_type": ["Protocol type", "نوع البروتوكول"],
    "duration": ["Packet lifetime (TTL)", "عمر الحزمة (TTL)"],
    "rate": ["Packet rate", "معدل إرسال الحزم"],
    "fin_flag_number": ["Connection-close (FIN) signals", "إشارات إنهاء الاتصال (FIN)"],
    "syn_flag_number": ["Connection-request (SYN) signals", "إشارات طلب الاتصال (SYN)"],
    "rst_flag_number": ["Connection-reset (RST) signals", "إشارات قطع الاتصال (RST)"],
    "psh_flag_number": ["Push (PSH) signals", "إشارات الدفع (PSH)"],
    "ack_flag_number": ["Acknowledgment (ACK) signals", "إشارات التأكيد (ACK)"],
    "ece_flag_number": ["Congestion (ECE) signals", "إشارات الازدحام (ECE)"],
    "cwr_flag_number": ["Congestion-reduced (CWR) signals", "إشارات تقليل الازدحام (CWR)"],
    "ack_count": ["Number of ACK packets", "عدد حزم التأكيد"],
    "syn_count": ["Number of SYN packets", "عدد حزم طلب الاتصال"],
    "fin_count": ["Number of FIN packets", "عدد حزم إنهاء الاتصال"],
    "rst_count": ["Number of RST packets", "عدد حزم قطع الاتصال"],
    "http": ["Web traffic (HTTP)", "ترافيك ويب (HTTP)"],
    "https": ["Secure web traffic (HTTPS)", "ترافيك ويب آمن (HTTPS)"],
    "dns": ["Address lookups (DNS)", "طلبات العناوين (DNS)"],
    "telnet": ["Remote access (Telnet)", "وصول عن بُعد (Telnet)"],
    "ssh": ["Secure remote access (SSH)", "وصول آمن عن بُعد (SSH)"],
    "tcp": ["Use of the TCP protocol", "استخدام بروتوكول TCP"],
    "udp": ["Use of the UDP protocol", "استخدام بروتوكول UDP"],
    "dhcp": ["Network setup (DHCP)", "إعداد الشبكة (DHCP)"],
    "arp": ["ARP messages (device addresses)", "رسائل ARP (عناوين الأجهزة)"],
    "icmp": ["Ping messages (ICMP)", "رسائل الفحص (ICMP)"],
    "igmp": ["Group messages (IGMP)", "رسائل المجموعات (IGMP)"],
    "ipv": ["Use of the IP protocol", "استخدام بروتوكول IP"],
    "tot_sum": ["Total size of packets", "مجموع أحجام الحزم"],
    "min": ["Smallest packet", "أصغر حزمة"],
    "max": ["Largest packet", "أكبر حزمة"],
    "avg": ["Average packet size", "متوسط حجم الحزم"],
    "std": ["Variation in packet size", "تفاوت أحجام الحزم"],
    "tot_size": ["Packet size", "حجم الحزمة"],
    "iat": ["Time between packets", "الوقت بين الحزم"],
    "number": ["Number of packets", "عدد الحزم"],
    "magnitue": ["Overall packet size (magnitude)", "الحجم العام للحزم"],
    "radius": ["Spread of packet sizes (radius)", "انتشار أحجام الحزم"],
    "covariance": ["In/out size relation (covariance)", "علاقة أحجام الداخل والخارج"],
    "variance": ["In/out size difference (variance)", "تباين أحجام الداخل والخارج"],
    "weight": ["Traffic weight", "وزن الترافيك"],
}

# what each traffic type means, in plain words
FLOOD_WHAT = {"SYN": ["connection requests", "طلبات اتصال"], "TCP": ["TCP packets", "حزم TCP"],
              "UDP": ["data packets (UDP)", "حزم بيانات UDP"], "ICMP": ["ping messages (ICMP)", "رسائل ping"]}
MQTT_HUB = ["the MQTT server (the messaging hub the devices use to send their readings)",
            "خادم MQTT (المركز اللي ترسل له الأجهزة قراءاتها)"]

ATTACKS = {"Benign": ["Normal traffic", "Everyday traffic from the medical devices, with no attack.",
                      "ترافيك طبيعي", "ترافيك يومي عادي من الأجهزة الطبية، بدون أي هجوم."]}
for kind, (en, ar) in FLOOD_WHAT.items():
    ATTACKS[f"DDoS {kind} Flood"] = [f"DDoS {kind} Flood",
        f"Many machines flood the device with {en} until it stops responding.", f"DDoS {kind} Flood",
        f"أجهزة كثيرة تغرق الجهاز بـ{ar} حتى يتوقف عن الاستجابة."]
    ATTACKS[f"DoS {kind} Flood"] = [f"DoS {kind} Flood",
        f"One machine floods the device with {en} to slow it down or crash it.", f"DoS {kind} Flood",
        f"جهاز واحد يغرق الجهاز بـ{ar} لإبطائه أو إيقافه."]
for who_en, who_ar, kind in [("Many machines", "أجهزة كثيرة", "DDoS"), ("One machine", "جهاز واحد", "DoS")]:
    ATTACKS[f"MQTT {kind} Connect Flood"] = [f"MQTT {kind} Connect Flood",
        f"{who_en} keep opening connections to {MQTT_HUB[0]} until it can't serve the real devices.",
        f"MQTT {kind} Connect Flood", f"{who_ar} تفتح اتصالات بشكل متواصل مع {MQTT_HUB[1]} حتى ما يقدر يخدم الأجهزة الحقيقية."]
    ATTACKS[f"MQTT {kind} Publish Flood"] = [f"MQTT {kind} Publish Flood",
        f"{who_en} flood {MQTT_HUB[0]} with fake messages until it can't keep up.",
        f"MQTT {kind} Publish Flood", f"{who_ar} تغرق {MQTT_HUB[1]} برسائل مزيفة حتى ما يقدر يلحق عليها."]
ATTACKS.update({
    "MQTT Malformed Data": ["MQTT Malformed Data",
        f"Sends broken, badly formed messages to {MQTT_HUB[0]} to confuse or crash it. In the numbers it looks a lot like normal traffic.",
        "MQTT Malformed Data", f"يرسل رسائل خربانة وغير سليمة إلى {MQTT_HUB[1]} عشان يلخبطه أو يوقفه. في الأرقام شكله قريب جدًا من الطبيعي."],
    "Recon PortScan": ["Recon PortScan", "Scans the device's ports to find open doors to attack later.",
        "Recon PortScan", "يفحص منافذ الجهاز عشان يلقى أبواب مفتوحة يهاجم منها بعدين."],
    "Recon OSScan": ["Recon OSScan", "Figures out which operating system the device runs, to choose the right attack.",
        "Recon OSScan", "يكتشف نظام التشغيل اللي يشتغل عليه الجهاز عشان يختار الهجوم المناسب."],
    "Recon VulScan": ["Recon VulScan", "Searches the device for known security weaknesses.",
        "Recon VulScan", "يدوّر في الجهاز على ثغرات أمنية معروفة."],
    "Recon PingSweep": ["Recon PingSweep", "Pings many addresses to find out which devices are online.",
        "Recon PingSweep", "يرسل ping لعناوين كثيرة عشان يعرف أي الأجهزة شغالة."],
    "ARP Spoofing": ["ARP Spoofing",
        "An attacker pretends to be another device on the network to secretly intercept its traffic. "
        "It looks almost like normal traffic, so it's very hard to catch.",
        "ARP Spoofing", "المهاجم ينتحل شخصية جهاز ثاني في الشبكة عشان يعترض بياناته بالسر. شكله قريب جدًا من الطبيعي، وعشان كذا صعب كشفه."],
})

missing = set(attack_test) - set(ATTACKS)
assert not missing, f"no description for {missing}"
assert set(FEATURES) == set(FEATURE_NAMES), "feature names do not match config.json"
print(f"Texts ready: {len(ATTACKS)} traffic types, {len(FEATURE_NAMES)} features, 2 languages")

## 4. The design

This cell has the two color palettes (dark and light), the icons (small SVG drawings), and the functions that build each part of the page as HTML:

- `hero_html`: the title, the 3 steps of how the model works, and the shield picture. The shield turns **blue with a check** for normal traffic and **red with "!"** for an attack
- `stats_html`: the 4 stat cards
- `result_html`: the result panel. It calls `predict()` for the chosen row and shows the decision, a gauge with the normal zone, the alarm line (the threshold) and the attack zone, what the row really was, and the 3 features with the biggest reconstruction error
- `page_css`: restyles Gradio's own buttons and inputs so they match the rest of the page

The gauge uses a log scale (from 1e-7 to 1e-1), because the scores of normal rows and attacks are thousands of times apart and would not fit on a normal scale.

In [ ]:
# colors, icons and the HTML pieces of the page
from urllib.parse import quote

PALETTE = {
    "dark": {
        "bg": "#060A14", "glow": "radial-gradient(900px 500px at 15% -5%, rgba(37,99,235,0.22), transparent 60%)",
        "card": "#0C1322", "inner": "#0A101D", "seg": "#111A2D", "border": "rgba(148,163,184,0.14)",
        "shadow": "0 24px 48px rgba(0,0,0,0.45)", "shadow_sm": "0 10px 24px rgba(0,0,0,0.35)",
        "text": "#E6EDF7", "sub": "#9AA8BD", "muted": "#6B7A90", "accent": "#60A5FA", "accent_soft": "rgba(59,130,246,0.16)",
        "pill": "rgba(34,197,94,0.10)", "pill_text": "#86EFAC", "pill_border": "rgba(34,197,94,0.25)",
        "steps": [("rgba(20,184,166,0.08)", "rgba(20,184,166,0.22)", "#5EEAD4"),
                  ("rgba(59,130,246,0.08)", "rgba(59,130,246,0.24)", "#93C5FD"),
                  ("rgba(249,115,22,0.08)", "rgba(249,115,22,0.22)", "#FDBA74")],
        "art": "#070D1B", "art_glow": "rgba(56,132,255,0.40)", "dot": "rgba(148,163,184,0.16)",
        "tile": "rgba(20,30,52,0.85)", "tile_icon": "#C7D2E4", "tile_shadow": "0 8px 18px rgba(0,0,0,0.35)",
        "input": "#0A101D", "input_border": "rgba(148,163,184,0.22)", "track": "rgba(148,163,184,0.14)",
        "zone_ok": "rgba(45,212,191,0.55)", "zone_bad": "rgba(251,113,133,0.55)", "ok_text": "#5EEAD4", "bad_text": "#FDA4AF",
        "good_box": ("rgba(34,197,94,0.09)", "rgba(34,197,94,0.30)", "#86EFAC"),
        "bad_box": ("rgba(244,63,94,0.10)", "rgba(244,63,94,0.32)", "#FDA4AF"),
        "chip_ok": ("rgba(34,197,94,0.14)", "#86EFAC"), "chip_miss": ("rgba(245,158,11,0.16)", "#FCD34D"),
        "ex": {"normal": ("rgba(20,184,166,0.12)", "#2DD4BF"), "easy": ("rgba(249,115,22,0.12)", "#FB923C"),
               "hard": ("rgba(139,92,246,0.14)", "#A78BFA")},
    },
    "light": {
        "bg": "#F3F6FC", "glow": "radial-gradient(900px 500px at 15% -5%, rgba(37,99,235,0.10), transparent 60%)",
        "card": "#FFFFFF", "inner": "#F7F9FC", "seg": "#FFFFFF", "border": "#E3E8F0",
        "shadow": "0 1px 2px rgba(16,24,40,0.05), 0 16px 36px rgba(16,24,40,0.07)",
        "shadow_sm": "0 1px 2px rgba(16,24,40,0.05), 0 6px 16px rgba(16,24,40,0.05)",
        "text": "#0F172A", "sub": "#4A5568", "muted": "#6B7280", "accent": "#2563EB", "accent_soft": "#EAF1FF",
        "pill": "#ECFDF3", "pill_text": "#067647", "pill_border": "#ABEFC6",
        "steps": [("#EFFCF8", "#C8F2E6", "#0B6B61"), ("#EFF5FF", "#D3E2FD", "#1E4FBF"), ("#FFF5EE", "#FCDDC8", "#9A3412")],
        "art": "#EEF3FC", "art_glow": "rgba(59,130,246,0.22)", "dot": "rgba(30,41,59,0.10)",
        "tile": "#FFFFFF", "tile_icon": "#334155", "tile_shadow": "0 6px 14px rgba(16,24,40,0.06)",
        "input": "#FFFFFF", "input_border": "#CFD6E2", "track": "#E6EAF1",
        "zone_ok": "#99E6D2", "zone_bad": "#FDB4BF", "ok_text": "#0B7A6E", "bad_text": "#BE123C",
        "good_box": ("#ECFDF3", "#ABEFC6", "#067647"), "bad_box": ("#FFF1F2", "#FECDD3", "#BE123C"),
        "chip_ok": ("#DCFAE6", "#067647"), "chip_miss": ("#FEF0C7", "#93370D"),
        "ex": {"normal": ("#ECFDF7", "#0D9488"), "easy": ("#FFF4ED", "#EA580C"), "hard": ("#F3EEFF", "#7C3AED")},
    },
}

ICONS = {
    "pulse": "<path d=\"M3 12h4l2-5 4 10 2-5h6\"/>",
    "wifi": "<path d=\"M5 12.5a10 10 0 0114 0\"/><path d=\"M8.5 16a5 5 0 017 0\"/><path d=\"M12 19.5h.01\"/>",
    "lock": "<rect x=\"5\" y=\"11\" width=\"14\" height=\"9\" rx=\"2\"/><path d=\"M8 11V8a4 4 0 018 0v3\"/>",
    "syringe": "<path d=\"M18 2l4 4\"/><path d=\"M17 7l-9.5 9.5-3 1 1-3L15 5\"/><path d=\"M14 4l6 6\"/><path d=\"M5.5 18.5L3 21\"/>",
    "thermo": "<path d=\"M14 14.76V4a2 2 0 10-4 0v10.76a4 4 0 104 0z\"/>",
    "cpu": "<rect x=\"6\" y=\"6\" width=\"12\" height=\"12\" rx=\"2\"/><path d=\"M9 2v4M15 2v4M9 18v4M15 18v4M2 9h4M2 15h4M18 9h4M18 15h4\"/>",
    "heart": "<path d=\"M12 20s-7-4.4-7-10a4 4 0 017-2.6A4 4 0 0119 10c0 5.6-7 10-7 10z\"/>",
    "server": "<rect x=\"4\" y=\"4\" width=\"16\" height=\"7\" rx=\"2\"/><rect x=\"4\" y=\"13\" width=\"16\" height=\"7\" rx=\"2\"/><path d=\"M8 7.5h.01M8 16.5h.01\"/>",
    "key": "<circle cx=\"8\" cy=\"15\" r=\"4\"/><path d=\"M11 12l9-9\"/><path d=\"M17 6l3 3\"/>",
    "bell": "<path d=\"M6 16v-5a6 6 0 0112 0v5l2 2H4z\"/><path d=\"M10 21h4\"/>",
    "radar": "<circle cx=\"12\" cy=\"12\" r=\"9\"/><circle cx=\"12\" cy=\"12\" r=\"4\"/><path d=\"M12 12l6-5\"/>",
    "shieldplus": "<path d=\"M12 3l7 3v5c0 4.5-3 8.3-7 10-4-1.7-7-5.5-7-10V6l7-3z\"/><path d=\"M12 9v6M9 12h6\"/>",
    "shieldok": "<path d=\"M12 3l7 3v5c0 4.5-3 8.3-7 10-4-1.7-7-5.5-7-10V6l7-3z\"/><path d=\"M8.5 12.5l2.3 2.3 4.7-5\"/>",
    "shieldx": "<path d=\"M12 3l7 3v5c0 4.5-3 8.3-7 10-4-1.7-7-5.5-7-10V6l7-3z\"/><path d=\"M12 8v5M12 16h.01\"/>",
    "brain": "<path d=\"M9 4a3 3 0 00-3 3v.5A3 3 0 004 10.5 3 3 0 005 13a3 3 0 001 5 3 3 0 003 2V4z\"/><path d=\"M15 4a3 3 0 013 3v.5a3 3 0 012 3 3 3 0 01-1 2.5 3 3 0 01-1 5 3 3 0 01-3 2V4z\"/>",
    "repeat": "<path d=\"M4 12a8 8 0 0114-5.3L20 9\"/><path d=\"M20 4v5h-5\"/><path d=\"M20 12a8 8 0 01-14 5.3L4 15\"/><path d=\"M4 20v-5h5\"/>",
    "alert": "<path d=\"M12 3l9.5 17h-19z\"/><path d=\"M12 10v4M12 17h.01\"/>",
    "target": "<circle cx=\"12\" cy=\"12\" r=\"9\"/><circle cx=\"12\" cy=\"12\" r=\"5\"/><circle cx=\"12\" cy=\"12\" r=\"1\"/>",
    "bolt": "<path d=\"M13 2L4 14h7l-1 8 9-12h-7l1-8z\"/>",
    "mask": "<path d=\"M12 3a9 9 0 100 18 9 9 0 000-18z\"/><path d=\"M9 10h.01M15 10h.01\"/><path d=\"M9 15c1.5-1 4.5-1 6 0\"/>",
    "search": "<circle cx=\"11\" cy=\"11\" r=\"7\"/><path d=\"M20 20l-3.5-3.5\"/>",
    "shuffle": "<path d=\"M16 3h5v5\"/><path d=\"M4 20L21 3\"/><path d=\"M21 16v5h-5\"/><path d=\"M15 15l6 6\"/><path d=\"M4 4l5 5\"/>",
    "sun": "<circle cx=\"12\" cy=\"12\" r=\"4\"/><path d=\"M12 2v2M12 20v2M4.9 4.9l1.4 1.4M17.7 17.7l1.4 1.4M2 12h2M20 12h2M4.9 19.1l1.4-1.4M17.7 6.3l1.4-1.4\"/>",
    "moon": "<path d=\"M20 14.5A8 8 0 019.5 4a8 8 0 1010.5 10.5z\"/>",
    "list": "<path d=\"M8 6h13M8 12h13M8 18h13M3 6h.01M3 12h.01M3 18h.01\"/>",
    "info": "<circle cx=\"12\" cy=\"12\" r=\"9\"/><path d=\"M12 11v5M12 8h.01\"/>",
    "chart": "<path d=\"M4 20V10M10 20V4M16 20v-7M22 20H2\"/>",
    "database": "<ellipse cx=\"12\" cy=\"5\" rx=\"8\" ry=\"3\"/><path d=\"M4 5v14c0 1.7 3.6 3 8 3s8-1.3 8-3V5\"/><path d=\"M4 12c0 1.7 3.6 3 8 3s8-1.3 8-3\"/>",
}


def icon(name, size=20, color="currentColor"):
    return (f"<svg aria-hidden=\"true\" width=\"{size}\" height=\"{size}\" viewBox=\"0 0 24 24\" fill=\"none\" "
            f"stroke=\"{color}\" stroke-width=\"1.8\" stroke-linecap=\"round\" stroke-linejoin=\"round\">{ICONS[name]}</svg>")


def icon_url(name, color):
    # an icon as a CSS background image (used inside the Gradio buttons)
    return "url(\"data:image/svg+xml;utf8," + quote(icon(name, 24, color).replace(" aria-hidden=\"true\"", "").replace("<svg ", "<svg xmlns=\"http://www.w3.org/2000/svg\" ")) + "\")"


SHIELD = "M100 8 L182 40 V100 C182 150 146 192 100 212 C54 192 18 150 18 100 V40 Z"
LOG_MIN, LOG_MAX = -7, -1          # the gauge goes from 1e-7 to 1e-1 (log scale)


def gauge_pos(score):
    return float(np.clip((np.log10(max(score, 1e-12)) - LOG_MIN) / (LOG_MAX - LOG_MIN) * 100, 3, 97))


def page_css(lang, dark, selected):
    # restyles Gradio's own components and the page; re-sent whenever the theme, language or selection changes
    c = PALETTE["dark" if dark else "light"]
    font = "'IBM Plex Sans Arabic', Inter, system-ui, sans-serif" if lang == "ar" else "Inter, system-ui, sans-serif"
    t = TEXT[lang]
    css = f"""
@import url("https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=IBM+Plex+Sans+Arabic:wght@400;500;600;700&display=swap");
body, gradio-app {{ background: {c["bg"]} !important; }}
.gradio-container {{
  --body-background-fill: {c["bg"]}; --background-fill-primary: {c["card"]}; --background-fill-secondary: {c["inner"]};
  --block-background-fill: transparent; --block-border-color: transparent; --border-color-primary: {c["border"]};
  --body-text-color: {c["text"]}; --body-text-color-subdued: {c["sub"]};
  --block-label-text-color: {c["sub"]}; --block-title-text-color: {c["sub"]}; --block-info-text-color: {c["muted"]};
  --input-background-fill: {c["input"]}; --input-background-fill-focus: {c["input"]}; --input-background-fill-hover: {c["input"]};
  --input-border-color: {c["input_border"]}; --input-border-color-focus: #2563EB; --input-border-color-hover: {c["input_border"]};
  --input-placeholder-color: {c["muted"]}; --input-shadow-focus: 0 0 0 3px rgba(37,99,235,0.25);
  --button-secondary-background-fill: {c["seg"]}; --button-secondary-background-fill-hover: {c["inner"]};
  --button-secondary-text-color: {c["text"]}; --button-secondary-border-color: {c["border"]};
  --button-primary-background-fill: #2563EB; --button-primary-background-fill-hover: #1D4ED8; --button-primary-text-color: #FFFFFF;
  --color-accent: #2563EB; --color-accent-soft: {c["accent_soft"]};
  --checkbox-label-background-fill: transparent; --checkbox-label-background-fill-hover: {c["inner"]};
  --checkbox-label-background-fill-selected: #2563EB; --checkbox-label-text-color: {c["sub"]};
  --checkbox-label-text-color-selected: #FFFFFF; --checkbox-label-border-color: transparent;
  background: {c["glow"]}, {c["bg"]} !important; color: {c["text"]}; font-family: {font} !important;
  direction: {"rtl" if lang == "ar" else "ltr"}; max-width: 1220px !important; margin: 0 auto !important;
}}
.gradio-container * {{ font-family: {font}; }}
footer {{ display: none !important; }}
.raw, .raw > div {{ padding: 0 !important; border: none !important; background: transparent !important; box-shadow: none !important; }}
#topbar {{ background: {c["card"]}; border: 1px solid {c["border"]}; border-radius: 18px; padding: 10px 14px; box-shadow: {c["shadow_sm"]}; align-items: center; }}
#topbar > * {{ align-self: center; }}
#lang-group {{ flex: 0 0 auto !important; width: auto !important; gap: 4px !important; padding: 4px; border-radius: 12px; background: {c["seg"]}; border: 1px solid {c["border"]}; }}
#lang-en, #lang-ar {{ flex: 0 0 auto !important; min-width: 64px !important; height: 36px; padding: 0 14px !important; border-radius: 9px !important;
  border: none !important; box-shadow: none !important; font-size: 14px !important; font-weight: 600 !important; background: transparent !important; color: {c["sub"]} !important; }}
#lang-{"ar" if lang == "ar" else "en"} {{ background: #2563EB !important; color: #FFFFFF !important; box-shadow: 0 4px 10px rgba(37,99,235,0.35) !important; }}
#theme-btn {{ flex: 0 0 auto !important; width: auto !important; min-width: 0 !important; height: 46px; padding: 0 16px !important; border-radius: 12px !important;
  font-size: 14px !important; font-weight: 500; display: flex; gap: 8px; align-items: center; justify-content: center; }}
#picker input, #picker input:focus, #picker input:hover {{ background: {c["input"]} !important; color: {c["text"]} !important; -webkit-text-fill-color: {c["text"]}; }}
#theme-btn::before {{ content: ""; width: 18px; height: 18px; background: {icon_url("sun" if dark else "moon", c["text"])} center / contain no-repeat; }}
#picker {{ background: {c["card"]}; border: 1px solid {c["border"]}; border-radius: 22px; padding: 22px; box-shadow: {c["shadow"]}; gap: 12px; }}
#picker .block {{ background: transparent !important; border: none !important; box-shadow: none !important; padding: 0 !important; }}
#picker input[type=number] {{ height: 46px; border-radius: 12px !important; font-size: 15px; }}
.ex-btn {{ display: grid !important; grid-template-columns: 40px 1fr; grid-template-rows: auto auto; column-gap: 12px;
  align-items: center; justify-items: start; text-align: start !important; min-height: 64px; padding: 10px 12px !important;
  border-radius: 14px !important; background: {c["inner"]} !important; border: 1px solid {c["border"]} !important;
  color: {c["text"]} !important; font-weight: 600 !important; font-size: 14px !important; box-shadow: none !important; }}
.ex-btn::before {{ content: ""; grid-row: 1 / 3; width: 40px; height: 40px; border-radius: 11px;
  background-repeat: no-repeat; background-position: center; background-size: 20px; }}
.ex-btn::after {{ font-size: 12px; font-weight: 400; color: {c["sub"]}; }}
#check-btn, #random-btn {{ height: 46px; border-radius: 12px !important; font-size: 15px !important; display: flex; gap: 8px; align-items: center; justify-content: center; }}
#check-btn {{ box-shadow: 0 8px 20px rgba(37,99,235,0.35) !important; font-weight: 600 !important; }}
#check-btn::before {{ content: ""; width: 18px; height: 18px; background: {icon_url("search", "#FFFFFF")} center / contain no-repeat; }}
#random-btn::before {{ content: ""; width: 18px; height: 18px; background: {icon_url("shuffle", c["text"])} center / contain no-repeat; }}
"""
    icons = {"normal": "pulse", "easy": "bolt", "hard": "mask"}
    for key, (tint, color) in c["ex"].items():
        sub = t["ex"][key][1]
        css += f"#ex-{key}::before {{ background-color: {tint}; background-image: {icon_url(icons[key], color)}; }}\n"
        css += f"#ex-{key}::after {{ content: \"{sub}\"; }}\n"
        if key == selected:
            css += (f"#ex-{key} {{ background: {tint} !important; border: 1.5px solid {color} !important; "
                    f"box-shadow: {c["shadow_sm"]} !important; }}\n")
    return f"<style>{css}</style>"


def card_style(c, radius=22, pad=22):
    return (f"background: {c["card"]}; border: 1px solid {c["border"]}; border-radius: {radius}px; "
            f"padding: {pad}px; box-shadow: {c["shadow"]};")


def brand_html(lang, dark):
    c, t = PALETTE["dark" if dark else "light"], TEXT[lang]
    return f"""<div dir="{"rtl" if lang == "ar" else "ltr"}" style="display: flex; align-items: center; gap: 12px; color: {c["text"]}">
<div style="width: 42px; height: 42px; border-radius: 12px; background: #2563EB; color: #FFFFFF; display: flex; align-items: center; justify-content: center; box-shadow: 0 6px 18px rgba(37,99,235,0.45)">{icon("shieldok", 22)}</div>
<div><div style="font-size: 17px; font-weight: 700">{t["title"]}</div><div style="font-size: 12px; color: {c["sub"]}">{t["tag"]}</div></div></div>"""


def hero_html(lang, dark, attack):
    c, t = PALETTE["dark" if dark else "light"], TEXT[lang]
    steps = ""
    for i, (ic, color) in enumerate([("brain", "#0D9488"), ("repeat", "#2563EB"), ("alert", "#EA580C")]):
        bg, border, title_color = c["steps"][i]
        steps += f"""<div style="display: flex; gap: 12px; align-items: center; padding: 10px 12px; border-radius: 14px; background: {bg}; border: 1px solid {border}">
<span style="flex: none; width: 38px; height: 38px; border-radius: 11px; display: flex; align-items: center; justify-content: center; background: {color}; color: #FFFFFF">{icon(ic, 20)}</span>
<span><b style="display: block; font-size: 14px; font-weight: 600; color: {title_color}">{i + 1} · {t[f"s{i + 1}h"]}</b><span style="font-size: 13px; color: {c["sub"]}">{t[f"s{i + 1}"]}</span></span></div>"""
    tiles = ["pulse", "wifi", "lock", "syringe", "thermo", "cpu", "heart", "server", "key", "bell", "radar", "shieldplus"]
    opacity = [1, 0.9, 0.9, 1, 0.75, 0.25, 0.25, 0.75, 0.55, 0.35, 0.35, 0.55]
    tile_html = "".join(
        f"""<div style="aspect-ratio: 1; border-radius: 16px; display: flex; align-items: center; justify-content: center; opacity: {o}; background: {c["tile"]}; border: 1px solid {c["border"]}; color: {c["tile_icon"]}; box-shadow: {c["tile_shadow"]}">{icon(n, 24)}</div>"""
        for n, o in zip(tiles, opacity))
    if attack:
        fills, stroke = ("rgba(244,63,94,0.10)", "rgba(244,63,94,0.18)", "rgba(225,29,72,0.78)"), "rgba(253,164,175,0.55)"
        mark = "<path d=\"M100 82 V118\" fill=\"none\" stroke=\"#FFFFFF\" stroke-width=\"11\" stroke-linecap=\"round\"/><circle cx=\"100\" cy=\"140\" r=\"6.5\" fill=\"#FFFFFF\"/>"
        drop = "rgba(225,29,72,0.45)"
    else:
        fills, stroke = ("rgba(96,165,250,0.10)", "rgba(96,165,250,0.18)", "rgba(37,99,235,0.78)"), "rgba(147,197,253,0.55)"
        mark = "<path d=\"M74 112 L93 131 L130 92\" fill=\"none\" stroke=\"#FFFFFF\" stroke-width=\"11\" stroke-linecap=\"round\" stroke-linejoin=\"round\"/>"
        drop = "rgba(37,99,235,0.45)"
    shield = "".join(
        f"<g transform=\"translate(100 110) scale({s}) translate(-100 -110)\"><path d=\"{SHIELD}\" fill=\"{f}\" stroke=\"{stroke}\" stroke-width=\"1.4\"/></g>"
        for s, f in zip([1, 0.82, 0.64], fills))
    return f"""<div dir="{"rtl" if lang == "ar" else "ltr"}" style="{card_style(c, 24, 28)} display: flex; flex-wrap: wrap; gap: 20px; color: {c["text"]}; overflow: hidden">
<div style="flex: 1 1 420px; min-width: 0; display: flex; flex-direction: column; gap: 18px; justify-content: center">
<span style="align-self: flex-start; display: inline-flex; align-items: center; gap: 8px; font-size: 12px; font-weight: 500; padding: 6px 12px; border-radius: 999px; background: {c["pill"]}; color: {c["pill_text"]}; border: 1px solid {c["pill_border"]}"><span style="width: 7px; height: 7px; border-radius: 50%; background: #22C55E; box-shadow: 0 0 0 4px rgba(34,197,94,0.18)"></span>{t["badge"]}</span>
<div style="font-size: 38px; line-height: 1.15; font-weight: 700; letter-spacing: -0.02em">{t["hero_title"]}</div>
<div style="font-size: 15px; line-height: 1.75; color: {c["sub"]}">{t["hero_sub"]}</div>
<div style="display: flex; flex-direction: column; gap: 10px">{steps}</div></div>
<div aria-hidden="true" style="flex: 1 1 380px; min-width: 0; min-height: 380px; position: relative; border-radius: 20px; overflow: hidden; background-color: {c["art"]}; background-image: radial-gradient(circle at 50% 50%, {c["art_glow"]}, transparent 62%), radial-gradient({c["dot"]} 1px, transparent 1.2px); background-size: 100% 100%, 16px 16px; border: 1px solid {c["border"]}; display: flex; align-items: center; justify-content: center; padding: 24px">
<div style="width: 100%; max-width: 380px; display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 14px">{tile_html}</div>
<div style="position: absolute; inset: 0; display: flex; align-items: center; justify-content: center; pointer-events: none">
<svg width="230" height="250" viewBox="0 0 200 220" style="filter: drop-shadow(0 18px 40px {drop})">{shield}{mark}</svg></div></div></div>"""


def stats_html(lang, dark):
    c, t = PALETTE["dark" if dark else "light"], TEXT[lang]
    values = [f"{STATS["f1"]:.3f}", f"{STATS["recall"]:.1%}", f"{STATS["false_alarms"]:.2%}", str(STATS["attack_types"])]
    tints = [("rgba(59,130,246,0.15)", "#3B82F6", "target"), ("rgba(20,184,166,0.15)", "#14B8A6", "radar"),
             ("rgba(249,115,22,0.15)", "#F97316", "bell"), ("rgba(139,92,246,0.16)", "#8B5CF6", "list")]
    cards = "".join(f"""<div style="display: flex; align-items: center; gap: 14px; padding: 18px; border-radius: 18px; background: {c["card"]}; border: 1px solid {c["border"]}; box-shadow: {c["shadow_sm"]}">
<span style="flex: none; width: 46px; height: 46px; border-radius: 13px; display: flex; align-items: center; justify-content: center; background: {bg}; color: {fg}">{icon(ic, 22)}</span>
<span><b dir="ltr" style="display: block; font-size: 26px; font-weight: 700; letter-spacing: -0.02em; color: {c["text"]}">{v}</b><span style="font-size: 13px; color: {c["sub"]}">{label}</span></span></div>"""
        for v, label, (bg, fg, ic) in zip(values, t["stats"], tints))
    return f"""<div dir="{"rtl" if lang == "ar" else "ltr"}" style="display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 14px">{cards}</div>"""


def picker_head_html(lang, dark):
    c, t = PALETTE["dark" if dark else "light"], TEXT[lang]
    return f"""<div dir="{"rtl" if lang == "ar" else "ltr"}" style="color: {c["text"]}">
<div style="display: flex; align-items: center; gap: 10px">
<span style="width: 36px; height: 36px; border-radius: 10px; display: flex; align-items: center; justify-content: center; background: {c["accent_soft"]}; color: {c["accent"]}">{icon("database", 19)}</span>
<div><div style="font-size: 17px; font-weight: 600">{t["pick"]}</div><div style="font-size: 12px; color: {c["sub"]}">{t["pick_sub"].format(n=N_ROWS)}</div></div></div>
<div style="margin-top: 16px; font-size: 12px; font-weight: 600; letter-spacing: 0.04em; color: {c["muted"]}">{t["try"]}</div></div>"""


def result_html(lang, dark, row, r=None, error=None):
    c, t = PALETTE["dark" if dark else "light"], TEXT[lang]
    d = "rtl" if lang == "ar" else "ltr"
    if error:
        return f"""<div dir="{d}" style="{card_style(c)} color: {c["text"]}"><div style="display: flex; gap: 12px; align-items: center; border-radius: 16px; padding: 16px 18px; background: {c["chip_miss"][0]}; color: {c["chip_miss"][1]}; font-weight: 600">{icon("info", 22)}{error}</div></div>"""

    score, attack, top = r["score"], r["is_anomaly"], r["top_features"]
    truth = attack_test[row]
    is_attack = truth != "Benign"
    ratio = score / THRESHOLD
    if ratio >= 2:
        ratio_text = t["far_above"].format(r=ratio)
    elif ratio > 1:
        ratio_text = t["above"]
    elif ratio > 0.5:
        ratio_text = t["below"]
    else:
        ratio_text = t["far_below"]

    box_bg, box_border, title_color = c["bad_box"] if attack else c["good_box"]
    icon_bg = "background: #E11D48; box-shadow: 0 8px 20px rgba(225,29,72,0.40)" if attack else "background: #16A34A; box-shadow: 0 8px 20px rgba(22,163,74,0.38)"
    if attack == is_attack:
        chip_text, (chip_bg, chip_fg) = t["ok"], c["chip_ok"]
    else:
        chip_text, (chip_bg, chip_fg) = (t["miss"] if is_attack else t["false_alarm"]), c["chip_miss"]

    lang_i = 0 if lang == "en" else 2
    name, desc = ATTACKS[truth][lang_i], ATTACKS[truth][lang_i + 1]
    errs = ERRORS[row]
    idx = [FEATURES.index(f) for f in top]
    top_err = errs[idx]
    bars = ""
    for f, e, color in zip(top, top_err, ["#3B82F6", "#14B8A6", "#F97316"]):
        width = max(4, round(e / top_err.max() * 100)) if top_err.max() > 0 else 4
        label = FEATURE_NAMES[f][0 if lang == "en" else 1]
        bars += f"""<div style="display: flex; flex-direction: column; gap: 5px">
<div style="display: flex; justify-content: space-between; gap: 8px; align-items: baseline"><span style="font-size: 13px; font-weight: 500">{label}</span><code dir="ltr" style="font-size: 11px; color: {c["muted"]}">{f}</code></div>
<div style="height: 8px; border-radius: 999px; overflow: hidden; background: {c["track"]}"><div style="height: 8px; border-radius: 999px; width: {width}%; background: {color}"></div></div></div>"""

    line = (np.log10(THRESHOLD) - LOG_MIN) / (LOG_MAX - LOG_MIN) * 100
    pos = gauge_pos(score)
    side = "right" if lang == "ar" else "left"
    shift = "50%" if lang == "ar" else "-50%"
    inner = f"border-radius: 16px; padding: 16px; background: {c["inner"]}; border: 1px solid {c["border"]}; display: flex; flex-direction: column;"
    return f"""<div dir="{d}" style="{card_style(c)} display: flex; flex-direction: column; gap: 18px; color: {c["text"]}">
<div style="display: flex; align-items: center; gap: 14px; flex-wrap: wrap; border-radius: 16px; padding: 16px 18px; background: {box_bg}; border: 1px solid {box_border}">
<div style="flex: none; width: 50px; height: 50px; border-radius: 14px; display: flex; align-items: center; justify-content: center; color: #FFFFFF; {icon_bg}">{icon("shieldx" if attack else "shieldok", 26)}</div>
<div style="min-width: 0"><div style="font-size: 22px; font-weight: 700; color: {title_color}">{t["att"] if attack else t["nor"]}</div><div style="font-size: 14px; color: {c["sub"]}">{t["att_sub"] if attack else t["nor_sub"]}</div></div>
<span style="margin-inline-start: auto; font-size: 12px; font-weight: 500; border-radius: 999px; padding: 5px 11px; white-space: nowrap; background: {c["seg"]}; color: {c["sub"]}; border: 1px solid {c["border"]}">{t["sample"]} #{row:,}</span></div>

<div style="display: flex; flex-direction: column; gap: 10px">
<div style="display: flex; justify-content: space-between; align-items: baseline; gap: 12px; flex-wrap: wrap">
<div style="font-size: 15px; font-weight: 600; display: flex; align-items: center; gap: 8px"><span style="color: {c["accent"]}; display: flex">{icon("chart", 18)}</span>{t["how"]}</div>
<div style="font-size: 14px; font-weight: 600; color: {c["bad_text"] if ratio > 1 else c["ok_text"]}">{ratio_text}</div></div>
<div style="position: relative; padding-top: 32px">
<div style="position: absolute; top: 0; {side}: {pos:.1f}%; transform: translateX({shift}); display: flex; flex-direction: column; align-items: center">
<span style="font-size: 11px; font-weight: 600; border-radius: 6px; padding: 3px 8px; white-space: nowrap; background: {c["text"]}; color: {c["bg"]}">{t["here"]}</span>
<span style="width: 2px; height: 11px; background: {c["text"]}"></span></div>
<div style="display: flex; height: 16px; border-radius: 999px; overflow: hidden">
<div style="width: {line:.1f}%; background: {c["zone_ok"]}"></div><div style="width: 3px; background: {c["text"]}"></div><div style="flex: 1; background: {c["zone_bad"]}"></div></div>
<div style="display: flex; justify-content: space-between; margin-top: 8px; font-size: 12px; font-weight: 500">
<span style="color: {c["ok_text"]}">{t["zone_ok"]}</span><span style="color: {c["sub"]}">{t["line"]}</span><span style="color: {c["bad_text"]}">{t["zone_bad"]}</span></div></div></div>

<div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 14px">
<div style="{inner} gap: 8px">
<div style="display: flex; align-items: center; justify-content: space-between; gap: 8px; flex-wrap: wrap">
<span style="font-size: 12px; font-weight: 600; letter-spacing: 0.04em; color: {c["muted"]}; display: flex; align-items: center; gap: 6px">{icon("info", 15)}{t["real"]}</span>
<span style="font-size: 12px; font-weight: 600; border-radius: 999px; padding: 4px 10px; background: {chip_bg}; color: {chip_fg}">{chip_text}</span></div>
<div style="font-size: 19px; font-weight: 700">{name}</div>
<div style="font-size: 13px; line-height: 1.7; color: {c["sub"]}">{desc}</div></div>
<div style="{inner} gap: 12px">
<span style="font-size: 12px; font-weight: 600; letter-spacing: 0.04em; color: {c["muted"]}; display: flex; align-items: center; gap: 6px">{icon("search", 15)}{t["feat"]}</span>{bars}</div></div>

<div dir="ltr" style="display: flex; flex-wrap: wrap; gap: 8px; font-size: 12px; color: {c["sub"]}">
<span style="border-radius: 8px; padding: 6px 10px; background: {c["inner"]}; border: 1px solid {c["border"]}">Reconstruction error: <b style="color: {c["text"]}">{score:.2e}</b></span>
<span style="border-radius: 8px; padding: 6px 10px; background: {c["inner"]}; border: 1px solid {c["border"]}">Threshold (P99): <b style="color: {c["text"]}">{THRESHOLD:.2e}</b></span></div></div>"""


def footer_html(lang, dark):
    c, t = PALETTE["dark" if dark else "light"], TEXT[lang]
    return f"""<div style="text-align: center; font-size: 12px; color: {c["muted"]}; padding: 4px 0 12px">{t["footer"]}</div>"""

## 5. The app

`render()` builds every part of the page for the current language, theme and sample, and every button calls it again with its change. The **Random sample** button picks a normal row half of the time and an attack the other half, because attacks are only 12% of the test set and we would almost never see one otherwise. If someone types a number that doesn't exist, the app shows a friendly message instead of crashing.

In [ ]:
import gradio as gr


def render(lang, dark, row, selected, error=None):
    # builds every part of the page for the current language, theme and sample
    t = TEXT[lang]
    r = None if error else predict(X_test[row])[0]          # the deployed inference pipeline
    attack = r["is_anomaly"] if r else bool(PRED[row])
    return (
        lang, dark, row, selected,
        page_css(lang, dark, selected),
        brand_html(lang, dark),
        gr.update(value=t["to_light"] if dark else t["to_dark"]),
        hero_html(lang, dark, attack=attack),
        stats_html(lang, dark),
        picker_head_html(lang, dark),
        gr.update(value=t["ex"]["normal"][0]),
        gr.update(value=t["ex"]["easy"][0]),
        gr.update(value=t["ex"]["hard"][0]),
        gr.update(label=t["num"].format(last=N_ROWS - 1), value=row),
        gr.update(value=t["check"]),
        gr.update(value=t["random"]),
        result_html(lang, dark, row, r, error),
        footer_html(lang, dark),
    )


def set_language(lang):
    return lambda dark, row, selected: render(lang, dark, row, selected)


def toggle_theme(lang, dark, row, selected):
    return render(lang, not dark, row, selected)


def show_example(key):
    return lambda lang, dark: render(lang, dark, EXAMPLES[key], key)


def check_sample(number, lang, dark, row, selected):
    if number is None or number != int(number) or not 0 <= number < N_ROWS:
        return render(lang, dark, row, selected, error=TEXT[lang]["bad_row"].format(last=N_ROWS - 1))
    row = int(number)
    selected = next((k for k, v in EXAMPLES.items() if v == row), None)
    return render(lang, dark, row, selected)


def random_sample(lang, dark, row, selected):
    # half the time a normal row, half the time an attack (attacks are only 12% of the test set)
    rows = BENIGN_ROWS if RNG.random() < 0.5 else ATTACK_ROWS
    return render(lang, dark, int(RNG.choice(rows)), None)


START = ("en", True, EXAMPLES["normal"], "normal")       # English, dark mode, the normal example
first = render(*START)
t0 = TEXT["en"]

with gr.Blocks() as demo:
    lang_state, dark_state, row_state, sel_state = (gr.State(v) for v in START)
    style = gr.HTML(first[4], elem_classes="raw")

    with gr.Row(elem_id="topbar", equal_height=False):
        brand = gr.HTML(first[5], elem_classes="raw")
        with gr.Row(elem_id="lang-group"):
            lang_en = gr.Button("EN", elem_id="lang-en", scale=0, min_width=64)
            lang_ar = gr.Button("عربي", elem_id="lang-ar", scale=0, min_width=64)
        theme_btn = gr.Button(t0["to_light"], elem_id="theme-btn", scale=0, min_width=0)

    hero = gr.HTML(first[7], elem_classes="raw")
    stats = gr.HTML(first[8], elem_classes="raw")

    with gr.Row(equal_height=False):
        with gr.Column(scale=4, min_width=320, elem_id="picker"):
            picker_head = gr.HTML(first[9], elem_classes="raw")
            ex_normal = gr.Button(t0["ex"]["normal"][0], elem_id="ex-normal", elem_classes="ex-btn")
            ex_easy = gr.Button(t0["ex"]["easy"][0], elem_id="ex-easy", elem_classes="ex-btn")
            ex_hard = gr.Button(t0["ex"]["hard"][0], elem_id="ex-hard", elem_classes="ex-btn")
            number = gr.Number(value=START[2], label=t0["num"].format(last=N_ROWS - 1),
                               precision=0, minimum=0, maximum=N_ROWS - 1)
            with gr.Row():
                check_btn = gr.Button(t0["check"], variant="primary", elem_id="check-btn")
                random_btn = gr.Button(t0["random"], variant="secondary", elem_id="random-btn")
        with gr.Column(scale=7, min_width=420):
            result = gr.HTML(first[16], elem_classes="raw")

    footer = gr.HTML(first[17], elem_classes="raw")

    STATES = [lang_state, dark_state, row_state, sel_state]
    OUTPUTS = STATES + [style, brand, theme_btn, hero, stats, picker_head,
                        ex_normal, ex_easy, ex_hard, number, check_btn, random_btn, result, footer]

    lang_en.click(set_language("en"), [dark_state, row_state, sel_state], OUTPUTS)
    lang_ar.click(set_language("ar"), [dark_state, row_state, sel_state], OUTPUTS)
    theme_btn.click(toggle_theme, STATES, OUTPUTS)
    ex_normal.click(show_example("normal"), [lang_state, dark_state], OUTPUTS)
    ex_easy.click(show_example("easy"), [lang_state, dark_state], OUTPUTS)
    ex_hard.click(show_example("hard"), [lang_state, dark_state], OUTPUTS)
    check_btn.click(check_sample, [number] + STATES, OUTPUTS)
    number.submit(check_sample, [number] + STATES, OUTPUTS)
    random_btn.click(random_sample, STATES, OUTPUTS)

demo.launch(share=True)

## 6. For the presentation

The plan is to try 3 rows live, in this order:

1. **Normal traffic:** the model says *Looks normal* and the marker is far inside the green zone.
2. **Easy attack (DDoS SYN Flood):** *Attack detected*, the score is far above the alarm line, and the top features are about connection signals, which makes sense for a flood of connection requests.
3. **Hard attack (ARP Spoofing):** the model says *Looks normal* and the chip says *Model missed it*. This is our biggest limitation: in our 40 features ARP Spoofing looks almost exactly like normal traffic (the model only catches about 2% of it), so a better model alone can't fix it. We would need different features, like how packets change over time.

Then we can use **Random sample** or the language and theme buttons to answer questions.